# Phase 1.1 — NASS Yield Data Ingestion

Fetch county-level corn yield history (2005–2024) for all 5 target states from the USDA NASS QuickStats API and cache the results to S3.

**States**: Iowa (IA), Colorado (CO), Wisconsin (WI), Missouri (MO), Nebraska (NE)  
**Metric**: CORN, GRAIN — YIELD, MEASURED IN BU/ACRE


In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
# If running from notebooks/ directory:
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from data_utils import fetch_all_states, load_all_states, STATES

BUCKET = "cornsight-data"


## Step 1 — Fetch from NASS API and cache to S3

Run this once. After the CSVs are in S3, use `load_all_states()` instead to avoid re-hitting the API.


In [ ]:
# NOTE: Set your NASS API key before running
# Either: os.environ["NASS_API_KEY"] = "your_key_here"
# Or set it in your shell: $env:NASS_API_KEY = "your_key"

yields_df = fetch_all_states(BUCKET, year_start=2005, year_end=2024)
print(f"\nTotal rows fetched: {len(yields_df)}")
print(yields_df["state"].value_counts())


## Step 2 — Reload from S3 (skip the API on subsequent runs)


In [ ]:
yields_df = load_all_states(BUCKET)
print(f"Loaded {len(yields_df)} rows from S3")
yields_df.head()


## Step 3 — Quick sanity checks


In [ ]:
import matplotlib.pyplot as plt

# Check year coverage per state
print("Year range per state:")
print(yields_df.groupby("state")["year"].agg(["min", "max", "count"]))

# Check for missing yield values
missing = yields_df["yield_bu_acre"].isna().sum()
print(f"\nMissing yield values: {missing} / {len(yields_df)}")

# State-level average yield over time
state_avg = (
    yields_df.groupby(["state", "year"])["yield_bu_acre"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for state, grp in state_avg.groupby("state"):
    ax.plot(grp["year"].astype(int), grp["yield_bu_acre"], marker="o", label=state)
ax.set_title("State-Average Corn Yield (bu/acre) — 2005–2024")
ax.set_xlabel("Year")
ax.set_ylabel("bu/acre")
ax.legend()
plt.tight_layout()
plt.show()


# Phase 1.2 — Cropland Data Layer (CDL)

Download CDL GeoTIFFs from USDA NASS CropScape for each state (2005–2024), mask to corn pixels only (class 1), and cache to S3.

`fetch_all_cdl()` automatically skips files already in S3, so it's safe to re-run if interrupted.


In [ ]:
from data_utils import fetch_all_cdl, load_cdl_from_s3, STATES

# Download CDL for all 5 states, 2005–2024, and cache to S3
# This will take a while — each state/year is ~50–200MB before masking
fetch_all_cdl(BUCKET, year_start=2005, year_end=2024)


In [ ]:
# Sanity check — load one state/year and confirm corn pixel count
corn_mask, profile = load_cdl_from_s3(BUCKET, state_fips="19", year=2023)  # Iowa 2023

print(f"Array shape : {corn_mask.shape}")
print(f"CRS         : {profile['crs']}")
print(f"Resolution  : {profile['transform'].a:.1f} m")
print(f"Corn pixels : {corn_mask.sum():,}")
print(f"Corn area   : {corn_mask.sum() * 30 * 30 / 1e6:.1f} km²")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(corn_mask, cmap="Greens", interpolation="none")
ax.set_title("Iowa 2023 — Corn Mask (CDL class 1)")
ax.axis("off")
plt.tight_layout()
plt.show()


# Phase 1.3 — HLS Multispectral Imagery

Download Harmonized Landsat Sentinel-2 (HLS) tiles from NASA EarthData for the corn growing season. These 6-band images are the direct input to Prithvi-100M.

**Required env vars** (set before running):
```
EARTHDATA_USERNAME=your_username
EARTHDATA_PASSWORD=your_password
```
Register free at https://urs.earthdata.nasa.gov/

**No download or S3 upload needed** — granules are streamed directly from NASA's S3 via `iter_granules_cloud()`. Only the extracted Prithvi feature vectors get written to your S3 bucket.

**Hackathon tip**: Use `max_granules=5` when testing to stream just a few tiles before running the full state/year.


In [ ]:
import sys
sys.path.insert(0, "../src")
from hls_downloader import login, iter_granules_cloud

login()

# Stream 5 granules for Iowa 2023 directly from NASA S3 — no download needed
for stacked, profile, gid in iter_granules_cloud("IA", 2023, "final", max_granules=5):
    print(f"  {gid}  shape={stacked.shape}  crs={profile['crs']}")
    for i, band in enumerate(["B02","B03","B04","B05","B06","B07"]):
        print(f"    {band}: min={stacked[i].min():.4f}  max={stacked[i].max():.4f}  mean={stacked[i].mean():.4f}")
    print()


In [ ]:
# Quick search count — confirms auth and bbox coverage before streaming
from hls_downloader import search_hls

for state in ["IA", "CO", "WI", "MO", "NE"]:
    results = search_hls(state, 2023, "final")
    print(f"{state} 2023 [final]: {len(results)} granules available")
